# Market Prices - Silver Layer Transformation

**Purpose:** Transform market price data from bronze to silver layer with cleaning, enrichment, and validation

**Bronze Table:** `vattenfall_dev.raw.bronze_market_prices`

**Silver Table:** `vattenfall_dev.refined.silver_market_prices`

**Transformations Applied:**
- Data cleaning and deduplication
- Type casting and validation of price ranges
- Temporal feature engineering (year, month, day, hour, weekend flags)
- Statistical calculations (rolling averages, price changes, volatility)
- Metadata enrichment (processing timestamps, quality scores)
- Outlier detection and flagging

In [0]:
import sys
import os

# Add project root to Python path
project_root = "/Workspace/Users/gharbi@startsteps.org/vattenfall-capstone-project"

if project_root not in sys.path:
    sys.path.append(project_root)


In [0]:
# Import transformation functions
from src.transforms import market_price_transforms

# Import validation functions
from src.validation.silver_checks import (
    SilverDataValidator,
    validate_market_prices
)

# Import PySpark functions
from pyspark.sql import functions as F


In [0]:
import yaml

# Load project configuration
config_path = f"{project_root}/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Extract configuration values
catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

# Define table names
bronze_table = f"{catalog}.{raw_schema}.bronze_market_prices"
silver_table = f"{catalog}.{refined_schema}.silver_market_prices"

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print(f"Catalog:        {catalog}")
print(f"Raw Schema:     {raw_schema}")
print(f"Refined Schema: {refined_schema}")
print(f"\nBronze Table:   {bronze_table}")
print(f"Silver Table:   {silver_table}")
print("=" * 70)

In [0]:
# Load bronze market prices data
df_bronze = spark.table(bronze_table)

print("=" * 70)
print("BRONZE DATA SUMMARY")
print("=" * 70)
print(f"Total Records: {df_bronze.count():,}")
print(f"Total Columns: {len(df_bronze.columns)}")
print("=" * 70)

print("\nBronze Schema:")
df_bronze.printSchema()

print("\nBronze Sample Data:")
display(df_bronze.limit(5))

In [0]:
# Apply complete transformation pipeline from bronze to silver
print("Starting transformation pipeline...\n")

df_silver = market_price_transforms.transform_bronze_to_silver(df_bronze)

# Drop unused columns (100% nulls from Bronze)
columns_to_drop = ["event_date", "region", "source_system", "last_updated_ts", "_rescued_data"]
existing_cols_to_drop = [col for col in columns_to_drop if col in df_silver.columns]

if existing_cols_to_drop:
    df_silver = df_silver.drop(*existing_cols_to_drop)
    print(f"✅ Dropped unused columns: {', '.join(existing_cols_to_drop)}")

print("✅ Transformations applied successfully")
print("\n" + "=" * 70)
print("TRANSFORMATION SUMMARY")
print("=" * 70)
print(f"Bronze Records:  {df_bronze.count():,}")
print(f"Silver Records:  {df_silver.count():,}")
print(f"Records Dropped: {df_bronze.count() - df_silver.count():,}")
print(f"\nBronze Columns:  {len(df_bronze.columns)}")
print(f"Silver Columns:  {len(df_silver.columns)}")
print("=" * 70)

In [0]:
print("Silver Layer Schema:")
df_silver.printSchema()

print("\n" + "=" * 70)
print("SILVER DATA SAMPLE")
print("=" * 70)

display(df_silver.limit(10))

print("\nNew Columns Added:")
new_columns = set(df_silver.columns) - set(df_bronze.columns)
for col in sorted(new_columns):
    print(f"  - {col}")

In [0]:
print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

# Record counts
total_silver = df_silver.count()
print(f"\nTotal Silver Records: {total_silver:,}")

# Completeness check for key columns
key_columns = ["timestamp", "market_zone", "price_eur_mwh", "price_24h_avg"]

print("\nCompleteness Check (Key Columns):")
for col in key_columns:
    if col in df_silver.columns:
        null_count = df_silver.filter(F.col(col).isNull()).count()
        completeness_pct = ((total_silver - null_count) / total_silver * 100) if total_silver > 0 else 0
        status = "✅" if completeness_pct == 100 else "⚠️"
        print(f"  {status} {col:20s}: {completeness_pct:6.2f}% complete ({null_count:,} nulls)")

# Check for outliers
if "is_price_outlier" in df_silver.columns:
    outlier_count = df_silver.filter(F.col("is_price_outlier") == True).count()
    outlier_pct = (outlier_count / total_silver * 100) if total_silver > 0 else 0
    print(f"\nPrice Outliers Detected: {outlier_count:,} ({outlier_pct:.2f}%)")

# Data quality score distribution
if "data_quality_score" in df_silver.columns:
    print("\nData Quality Score Distribution:")
    df_silver.groupBy("data_quality_score").count().orderBy("data_quality_score", ascending=False).show()

print("=" * 70)

In [0]:
print(f"Writing silver data to {silver_table}...\n")

# Write to silver table with Delta format
df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print("✅ Data successfully written to silver table")
print("\n" + "=" * 70)
print("WRITE OPERATION SUMMARY")
print("=" * 70)
print(f"Table:   {silver_table}")
print(f"Format:  Delta")
print(f"Mode:    Overwrite")
print(f"Records: {df_silver.count():,}")
print("=" * 70)

In [0]:
print("Running data quality validation...\n")

# Load silver table for validation
df_silver_verify = spark.table(silver_table)

# Run pre-configured validation
print("=" * 70)
print("AUTOMATED VALIDATION CHECKS")
print("=" * 70)
results = validate_market_prices(df_silver_verify)

# Run custom validation with detailed report
print("\n" + "=" * 70)
print("DETAILED VALIDATION REPORT")
print("=" * 70)

validator = SilverDataValidator(df_silver_verify, silver_table)

# Completeness check
validator.check_completeness(["timestamp", "market_zone", "price_eur_mwh", "price_24h_avg"])

# Duplicate check
validator.check_duplicates(["timestamp", "market_zone"])

# Value range checks
validator.check_value_ranges({
    "price_eur_mwh": (0, 1000),
    "price_24h_avg": (0, 1000),
    "hour": (0, 23),
    "month": (1, 12)
})

# Timeliness check
validator.check_timeliness("timestamp", max_age_hours=48)

# Generate and print report
print(validator.generate_report())

In [0]:
# Final verification and statistics
df_final = spark.table(silver_table)

print("=" * 70)
print("SILVER TABLE VERIFICATION")
print("=" * 70)
print(f"Table Name:    {silver_table}")
print(f"Total Records: {df_final.count():,}")
print(f"Total Columns: {len(df_final.columns)}")
print("=" * 70)

# Show descriptive statistics for price columns
print("\nPrice Statistics:")
df_final.select(
    "price_eur_mwh",
    "price_24h_avg",
    "price_change_hourly",
    "price_change_pct"
).describe().show()

# Show temporal distribution
print("\nTemporal Distribution (by hour category):")
df_final.groupBy("hour_category").count().orderBy("count", ascending=False).show()

print("\nMarket Zone Distribution:")
df_final.groupBy("market_zone").count().orderBy("count", ascending=False).show()

print("\n" + "=" * 70)
print("FINAL SAMPLE DATA")
print("=" * 70)
display(df_final.orderBy(F.desc("timestamp")).limit(20))

print("\n✅ Silver layer transformation complete!")
print(f"\nSilver table is ready for analytics: {silver_table}")

In [0]:
# Summary of key transformations applied
print("=" * 70)
print("KEY TRANSFORMATION RESULTS")
print("=" * 70)

# Region normalization results
print("\n1. REGION NORMALIZATION")
print("-" * 70)
df_final.groupBy("region_normalized", "market_zone") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("region_normalized", "market_zone") \
    .show(20, False)

# Market type normalization results
print("\n2. MARKET TYPE NORMALIZATION")
print("-" * 70)
df_final.groupBy("market_type", "market_type_normalized") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("market_type_normalized") \
    .show(10, False)

# Report day summary
print("\n3. REPORT DAY COVERAGE")
print("-" * 70)
df_final.groupBy("report_day") \
    .agg(
        F.count("*").alias("hourly_records"),
        F.countDistinct("market_zone").alias("zones"),
        F.round(F.avg("price_eur_mwh"), 2).alias("avg_price")
    ) \
    .orderBy("report_day") \
    .show(10, False)

# Price category distribution
print("\n4. PRICE CATEGORY DISTRIBUTION")
print("-" * 70)
df_final.groupBy("price_category") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("price_category") \
    .show()

print("\n" + "=" * 70)
print("✅ All transformations validated successfully!")
print("=" * 70)